<a href="https://colab.research.google.com/github/lucaslezcano03/Proyecto-IA/blob/main/football_players.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente 1: Normalizador de Datos
Este agente se encarga de la carga, limpieza, codificación y escalado del dataset `players.csv`.

## Resultados del Procesamiento
Aquí se muestra la confirmación de la generación del archivo y una previsualización de los datos finales.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

class AgenteNormalizador:
    def __init__(self, input_path, output_path):
        self.input_path = input_path
        self.output_path = output_path
        self.df = None

    def procesar(self):
        # 1. Carga
        self.df = pd.read_csv(self.input_path)

        # 2. Limpieza e Imputación
        # Columnas numéricas: Mediana
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if self.df[col].isnull().any():
                self.df[col] = self.df[col].fillna(self.df[col].median())

        # Columnas categóricas: 'Unknown'
        categorical_cols = ['foot', 'sub_position', 'position', 'domestic_league_code']
        for col in categorical_cols:
            if col in self.df.columns:
                self.df[col] = self.df[col].fillna('Unknown')

        # 3. Codificación (One-Hot Encoding)
        # Seleccionamos las columnas para encoding que el usuario especificó
        cols_to_encode = [c for c in ['position', 'foot', 'domestic_league_code'] if c in self.df.columns]
        self.df = pd.get_dummies(self.df, columns=cols_to_encode)

        # 4. Escalado (StandardScaler)
        # Escalamos columnas clave como market_value, height y age si existen
        cols_to_scale = [c for c in ['market_value_in_eur', 'height_in_cm', 'age'] if c in self.df.columns]
        scaler = StandardScaler()
        if cols_to_scale:
            self.df[cols_to_scale] = scaler.fit_transform(self.df[cols_to_scale])

        # 5. Exportación
        # Eliminamos columnas de texto restantes que no fueron procesadas para asegurar compatibilidad con ML
        self.df = self.df.select_dtypes(include=[np.number, 'bool'])
        self.df.to_csv(self.output_path, index=False)

        print('✓ Agente 1: Dataset procesado y players_clean.csv generado correctamente')

# Ejecución del Agente
normalizador = AgenteNormalizador('/content/players.csv', 'players_clean.csv')
normalizador.procesar()

# Mostrar previsualización
print("\nVista previa del dataset normalizado:")
display(pd.read_csv('players_clean.csv').head())

✓ Agente 1: Dataset procesado y players_clean.csv generado correctamente

Vista previa del dataset normalizado:


,player_id,last_season,current_club_id,height_in_cm,international_caps,international_goals,current_national_team_id,market_value_in_eur,highest_market_value_in_eur,position_Attack,position_Defender,position_Goalkeeper,position_Midfield,position_Missing,foot_Unknown,foot_both,foot_left,foot_right
0,10,2015,398,0.277582,4.0,0.0,5674.0,-0.028541,30000000.0,True,False,False,False,False,False,False,False,True
1,26,2017,16,1.119686,4.0,0.0,5674.0,-0.076204,8000000.0,False,False,True,False,False,False,False,True,False
2,65,2015,1091,-0.003119,4.0,0.0,5674.0,-0.028541,34500000.0,True,False,False,False,False,True,False,False,False
3,77,2012,506,-0.003119,4.0,0.0,5674.0,-0.181063,24500000.0,False,True,False,False,False,True,False,False,False
4,80,2017,27,1.681088,4.0,0.0,5674.0,-0.200129,3000000.0,False,False,True,False,False,False,False,False,True


## Agente 2: Entrenador
Este agente entrena modelos de regresión para predecir el valor de mercado de los jugadores y selecciona el mejor rendimiento.

In [2]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

class AgenteEntrenador:
    def __init__(self, data_path):
        self.data_path = data_path
        self.df = None
        self.X_train, self.X_test, self.y_train, self.y_test = [None] * 4
        self.results = {}

    def preparar_datos(self):
        self.df = pd.read_csv(self.data_path)
        y = self.df['market_value_in_eur']
        X = self.df.drop(columns=['market_value_in_eur'])
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

    def entrenar_y_evaluar(self):
        modelos = {
            'LinearRegression': LinearRegression(),
            'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42)
        }

        for nombre, modelo in modelos.items():
            # Entrenamiento
            modelo.fit(self.X_train, self.y_train)
            # Predicción
            preds = modelo.predict(self.X_test)
            # Métricas
            r2 = r2_score(self.y_test, preds)
            mae = mean_absolute_error(self.y_test, preds)

            self.results[nombre] = {'R2': r2, 'MAE': mae}

    def seleccionar_y_guardar(self):
        # Determinar el ganador basado en R2
        mejor_modelo = max(self.results, key=lambda x: self.results[x]['R2'])

        metadata = {
            'mejor_modelo': mejor_modelo,
            'metricas_ganador': self.results[mejor_modelo],
            'comparativa_completa': self.results
        }

        with open('resultados_entrenamiento.json', 'w') as f:
            json.dump(metadata, f, indent=4)

        # Mostrar tabla comparativa
        print("--- Comparativa de Modelos ---")
        print(pd.DataFrame(self.results).T)
        print(f"\nModelo Ganador: {mejor_modelo}")
        print('✓ Agente 2: Modelos entrenados y evaluados correctamente')

    def ejecutar_flujo(self):
        self.preparar_datos()
        self.entrenar_y_evaluar()
        self.seleccionar_y_guardar()

# Inicialización y ejecución
entrenador = AgenteEntrenador('players_clean.csv')
entrenador.ejecutar_flujo()

--- Comparativa de Modelos ---
                        R2       MAE
LinearRegression  0.627678  0.223411
RandomForest      0.848801  0.079233

Modelo Ganador: RandomForest
✓ Agente 2: Modelos entrenados y evaluados correctamente
